# Task 2 - API REST - RAWG Video Games Database

## Configuracion
Se carga `RAWG_API_KEY` desde `.env` y se define un cliente con contador de requests.

In [1]:
import os
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from dotenv import load_dotenv

# Buscar .env subiendo directorios hasta la raiz
env_loaded = False
for p in [Path.cwd(), *Path.cwd().parents]:
    candidate = p / '.env'
    if candidate.exists():
        load_dotenv(candidate)
        env_loaded = True
        break

API_KEY = os.getenv('RAWG_API_KEY')
if not API_KEY:
    raise ValueError('No se encontro RAWG_API_KEY en .env')

BASE_URL = 'https://api.rawg.io/api'

class RAWGClient:
    def __init__(self, api_key: str, timeout: int = 30):
        self.api_key = api_key
        self.timeout = timeout
        self._requests = 0

    def _get(self, endpoint: str, **params: Any) -> dict:
        url = f"{BASE_URL}/{endpoint}"
        query = {'key': self.api_key, **params}
        r = requests.get(url, params=query, timeout=self.timeout)
        self._requests += 1
        r.raise_for_status()
        return r.json()

    def games(self, **params: Any) -> dict:
        return self._get('games', **params)

    def game_detail(self, game_id: int) -> dict:
        return self._get(f'games/{game_id}')

    def resumen_requests(self) -> int:
        return self._requests

client = RAWGClient(API_KEY)
print('Cliente RAWG listo')

Cliente RAWG listo


## Parte A - General Exploration (2 pts)

In [2]:
resp_total = client.games(page_size=1)
total_games = resp_total.get('count', 0)
print(f"A1) Total de juegos registrados en RAWG: {total_games:,}")

A1) Total de juegos registrados en RAWG: 898,197


## Parte B - Category Analysis (2 pts)

In [3]:
resp_top_meta = client.games(ordering='-metacritic', page_size=5)
top5_meta = pd.DataFrame(resp_top_meta['results'])[['name', 'rating', 'metacritic']]
print('B1) Top 5 por metacritic')
top5_meta

B1) Top 5 por metacritic


,name,rating,metacritic
0,The Legend of Zelda: Ocarina of Time,4.38,99
1,Soulcalibur (1998),0.00,98
2,Soulcalibur,4.38,98
3,Baldur's Gate III,4.44,97
4,Metroid Prime,4.35,97


In [4]:
resp_steam = client.games(stores=1, ordering='-rating', page_size=10)
top10_steam = pd.DataFrame(resp_steam['results'])[['name', 'rating', 'metacritic']]
print('B2) Top 10 en Steam por rating')
top10_steam

B2) Top 10 en Steam por rating


,name,rating,metacritic
0,"Warhammer 40,000: Dawn of War - Definitive Edi...",4.83,NaN
1,No Case Should Remain Unsolved,4.83,NaN
2,The Witcher 3: Wild Hunt – Blood and Wine,4.81,92.0
3,The Witcher 3 Wild Hunt - Complete Edition,4.80,92.0
4,The Witcher 3: Wild Hunt – Hearts of Stone,4.76,90.0
5,Persona 5 Royal,4.75,94.0
6,Guilty Parade,4.71,NaN
7,Cyberpunk 2077: Phantom Liberty,4.71,NaN
8,The Binding of Isaac: Repentance,4.69,NaN
9,Red Matter 2,4.67,NaN


## Parte C - Comparisons (3 pts)

In [5]:
pc_resp = client.games(platforms=4, ordering='-rating', page_size=5)
ps5_resp = client.games(platforms=187, ordering='-rating', page_size=5)

pc_top5 = pd.DataFrame(pc_resp['results'])[['name', 'rating', 'metacritic']]
ps5_top5 = pd.DataFrame(ps5_resp['results'])[['name', 'rating', 'metacritic']]

pc_avg = pc_top5['rating'].mean()
ps5_avg = ps5_top5['rating'].mean()
best_platform = 'PC' if pc_avg > ps5_avg else 'PS5'

print(f"C1) Promedio rating PC: {pc_avg:.3f}")
print(f"C1) Promedio rating PS5: {ps5_avg:.3f}")
print(f"C1) Plataforma con mejor rating promedio: {best_platform}")

print('\nTop 5 PC')
display(pc_top5)
print('Top 5 PS5')
display(ps5_top5)

C1) Promedio rating PC: 4.826
C1) Promedio rating PS5: 4.724
C1) Plataforma con mejor rating promedio: PC

Top 5 PC


,name,rating,metacritic
0,The Elder Scrolls VI,4.86,NaN
1,"Warhammer 40,000: Dawn of War - Definitive Edi...",4.83,NaN
2,No Case Should Remain Unsolved,4.83,NaN
3,The Witcher 3: Wild Hunt – Blood and Wine,4.81,92.0
4,The Witcher 3 Wild Hunt - Complete Edition,4.80,92.0


Top 5 PS5


,name,rating,metacritic
0,The Witcher 3 Wild Hunt - Complete Edition,4.80,92.0
1,Persona 5 Royal,4.75,94.0
2,Cyberpunk 2077: Phantom Liberty,4.71,NaN
3,The Binding of Isaac: Repentance,4.69,NaN
4,The Last of Us Part I,4.67,NaN


In [6]:
famous_titles = ['The Witcher 3: Wild Hunt', 'Grand Theft Auto V', 'Elden Ring']
rows = []
for title in famous_titles:
    result = client.games(search=title, page_size=1).get('results', [])
    if not result:
        continue
    g = result[0]
    rows.append({
        'name': g.get('name'),
        'rating': g.get('rating'),
        'metacritic': g.get('metacritic'),
        'genres': ', '.join(x['name'] for x in g.get('genres', [])),
        'platforms': ', '.join(x['platform']['name'] for x in g.get('platforms', [])),
    })

c2_df = pd.DataFrame(rows)
print('C2) Comparacion de 3 juegos famosos')
c2_df

C2) Comparacion de 3 juegos famosos


,name,rating,metacritic,genres,platforms
0,The Witcher 3: Wild Hunt,4.64,92,"Action, RPG","PC, PlayStation 5, Xbox One, PlayStation 4, Xb..."
1,Grand Theft Auto V,4.47,92,Action,"PC, PlayStation 5, Xbox One, PlayStation 4, Xb..."
2,Elden Ring,4.38,95,"Action, RPG","PC, PlayStation 5, Xbox One, PlayStation 4, Xb..."


In [7]:
genre_map = {'Action': 4, 'Shooter': 2, 'RPG': 5, 'Strategy': 10}
rows = []
for genre_name, genre_id in genre_map.items():
    result = client.games(genres=genre_id, ordering='-rating', page_size=5)
    gdf = pd.DataFrame(result['results'])[['name', 'rating', 'metacritic']]
    rows.append({'genre': genre_name, 'avg_rating_top5': gdf['rating'].mean()})

c3_df = pd.DataFrame(rows).sort_values('avg_rating_top5', ascending=False).reset_index(drop=True)
print(f"C3) Genero con mejor promedio: {c3_df.iloc[0]['genre']}")
c3_df

C3) Genero con mejor promedio: RPG


,genre,avg_rating_top5
0,RPG,4.796
1,Action,4.772
2,Strategy,4.710
3,Shooter,4.684


In [8]:
years = [2015, 2018, 2023]
rows = []
for y in years:
    result = client.games(dates=f'{y}-01-01,{y}-12-31', ordering='-metacritic', page_size=20)
    ydf = pd.DataFrame(result['results'])[['name', 'rating', 'metacritic']]
    rows.append({'year': y, 'avg_metacritic_top20': ydf['metacritic'].dropna().mean()})

c4_df = pd.DataFrame(rows).sort_values('avg_metacritic_top20', ascending=False).reset_index(drop=True)
print(f"C4) Año con mayor promedio de metacritic: {int(c4_df.iloc[0]['year'])}")
c4_df

C4) Año con mayor promedio de metacritic: 2015


,year,avg_metacritic_top20
0,2015,90.4
1,2018,90.0
2,2023,83.2


In [9]:
out_dir = Path('output_task2')
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / 'top20_rawg.csv'

resp_top20 = client.games(ordering='-metacritic', page_size=20)
rows = []
for g in resp_top20['results']:
    rows.append({
        'name': g.get('name'),
        'rating': g.get('rating'),
        'metacritic': g.get('metacritic'),
        'release_date': g.get('released'),
        'main_genre': g.get('genres', [{}])[0].get('name') if g.get('genres') else None,
    })

top20_df = pd.DataFrame(rows)
top20_df.to_csv(out_csv, index=False, encoding='utf-8')
print(f"C5) CSV generado en: {out_csv.resolve()}")
print('Primeras 5 filas:')
pd.read_csv(out_csv).head()

C5) CSV generado en: C:\Users\johnb\Documents\GitHub\Tasks_Data_Science_2026_I\HW_01_202601\Scraping_data\api_task_2\output_task2\top20_rawg.csv
Primeras 5 filas:


,name,rating,metacritic,release_date,main_genre
0,The Legend of Zelda: Ocarina of Time,4.38,99,1998-11-21,Action
1,Soulcalibur (1998),0.00,98,1998-07-30,Fighting
2,Soulcalibur,4.38,98,1998-07-30,Action
3,Baldur's Gate III,4.44,97,2023-08-03,Adventure
4,Metroid Prime,4.35,97,2002-11-17,Action


## Parte D - Insights y Conclusiones (3 pts)

In [10]:
print(f"Requests totales usadas: {client.resumen_requests()}")

Requests totales usadas: 16


### D1 - Conclusiones personales

- Lo mas interesante es que los juegos top en metacritic no siempre son los top en rating de usuarios.
- Me sorprendio la competitividad de PS5, pero en varios cortes PC conserva mejor promedio.
- Si tuviera mas tiempo, analizaria como cambia la calidad promedio por genero a traves de los años.
- El total de requests usadas se reporta en la celda anterior con `client.resumen_requests()`.
